In [134]:
import numpy as np
from scipy.stats import linregress
import matplotlib.pyplot as plt
 
global_step = np.array([1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000, 13000, 14000, 15000, 16000, 17000, 18000, 19000, 20000, 21000, 22000, 23000, 24000, 25000, 26000, 27000, 28000, 29000, 30000, 31000, 32000, 33000, 34000, 35000, 36000, 37000, 38000, 39000, 40000, 41000, 42000, 43000, 44000, 45000, 46000, 47000, 48000, 49000, 50000])

# GBN
mse_adam = np.array([0.0060,0.0036,0.0023,0.0018,0.0022,0.0017,0.0035,0.0033,0.0017,0.0014,0.0019,0.0013,0.0044,0.0032,0.0017,0.0015,0.0019,0.0016,0.0017,0.0031,0.0014,0.0011,0.0027,0.0021,0.0040,0.0013,0.0012,0.0030,0.0027,0.0014,0.0011,0.0010,0.0050,0.0010,0.0010,0.0016,0.0014,0.0016,0.0017,0.0024,0.0011,0.0013,0.0028,0.0012,0.0011,0.0010,0.0017,0.0014,0.0013,0.0026])
mse_lamb = np.array([0.0094,0.0026,0.0008,0.0013,0.0013,0.0006,0.0010,0.0005,0.0002,0.0015,0.0003,0.0005,0.0002,0.0003,0.0002,0.0008,0.0002,0.0005,0.0002,0.0002,0.0005,0.0002,0.0003,0.0002,0.0002,0.0002,0.0008,0.0002,0.0003,0.0002,0.0003,0.0003,0.0002,0.0003,0.0001,0.0001,0.0001,0.0001,0.0001,0.0002,0.0002,0.0002,0.0003,0.0002,0.0001,0.0003,0.0002,0.0004,0.0003,0.0001])

# NSFNet
# mse_adam = np.array([0.0102,0.0061,0.0046,0.0054,0.0037,0.0040,0.0032,0.0021,0.0040,0.0039,0.0032,0.0025,0.0036,0.0026,0.0021,0.0033,0.0052,0.0033,0.0015,0.0096,0.0019,0.0019,0.0024,0.0034,0.0020,0.0029,0.0052,0.0017,0.0031,0.0028,0.0025,0.0027,0.0037,0.0015,0.0017,0.0045,0.0017,0.0018,0.0033,0.0021,0.0056,0.0016,0.0045,0.0027,0.0031,0.0026,0.0053,0.0050,0.0052,0.0022])
# mse_lamb = np.array([0.0110,0.0020,0.0009,0.0010,0.0027,0.0007,0.0020,0.0005,0.0006,0.0005,0.0013,0.0005,0.0010,0.0004,0.0007,0.0008,0.0004,0.0003,0.0008,0.0004,0.0004,0.0003,0.0003,0.0003,0.0007,0.0003,0.0003,0.0006,0.0003,0.0002,0.0012,0.0002,0.0003,0.0001,0.0003,0.0003,0.0002,0.0002,0.0004,0.0003,0.0003,0.0002,0.0003,0.0003,0.0003,0.0002,0.0002,0.0006,0.0003,0.0002])


In [135]:
# Parameters
threshold = 0.001  # Threshold for stability
window_size = 3   # Window size for smoothing the rate of change
sustained_steps = 3  # Number of consecutive windows to confirm plateau
 
# Function to smooth data and find plateau index
def find_plateau_with_cumulative_check(mse, steps, threshold, window_size, sustained_steps):
    
    # Smooth the MSE values
    smoothed_mse = np.convolve(mse, np.ones(window_size)/window_size, mode='valid')
    
    # Compute rate of change
    rate_of_change = np.abs(np.diff(smoothed_mse))
    
    # Check when cumulative average rate of change stays below threshold
    for i in range(len(rate_of_change) - sustained_steps + 1):
        if np.all(rate_of_change[i:i + sustained_steps] < threshold):
            return steps[i + window_size + sustained_steps - 1]  # Map to global step
            
    return None  # No plateau found
 
# Detect plateaus for both models
adam_plateau_step = find_plateau_with_cumulative_check(mse_adam, global_step, threshold, window_size, sustained_steps)
lamb_plateau_step = find_plateau_with_cumulative_check(mse_lamb, global_step, threshold, window_size, sustained_steps)
 
# Output
print("adam Plateau Step:", adam_plateau_step)
print("lamb Plateau Step:", lamb_plateau_step)

adam Plateau Step: 7000
lamb Plateau Step: 7000


In [136]:
def convergence_stability(global_step, mse_values, p, q):
    
    start_index = np.where(global_step >= p)[0][0]
    end_index = np.where(global_step <= q)[0][-1] + 1
    window_values = mse_values[start_index:end_index]
    window_mean = np.mean(window_values)
    abs_differences = np.abs(window_values - window_mean)
    stability = np.mean(abs_differences)
    return np.round(stability,4)
 
adam_convergence_point = 7000
p, q = 7000, 50000  # Example single window from 7k to 50k
adam_convergence_stability = convergence_stability(global_step, mse_adam, p, q)
print("adam Stability:", adam_convergence_stability)


lamb_convergence_point = 7000
p, q = 7000, 50000  # Example single window from 7k to 50k
lamb_convergence_stabiloity = convergence_stability(global_step, mse_lamb, p, q)
print("lamb Stability:", lamb_convergence_stabiloity)


adam Stability: 0.0008
lamb Stability: 0.0002
